In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import os
import re
import torch.utils.data as data

In [2]:
import os
import torch
from torch.utils.data import Dataset
import string
import unicodedata

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)


def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        # Исправил опечатку: добавил пробел перед and
        if unicodedata.category(c) != 'Mn' and c in all_letters
    )
    
    
class NameDataset(Dataset):
    def __init__(self, data_dir='names'):
        self.names = []
        self.labels = []
        self.nationalities = []
        
        self.files = sorted([f for f in os.listdir(data_dir) if f.endswith('.txt')])
        self.classes = [filename[:-4] for filename in self.files]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        for filename in self.files:
            nationality = filename[:-4]  # убираем .txt
            label = self.class_to_idx[nationality]
                
            filepath = os.path.join(data_dir, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                for line in f:
                    name = unicode_to_ascii(line.strip())
                    if name:  # пропускаем пустые строки
                        self.names.append(name)
                        self.labels.append(label)
    
    def __len__(self):
        return len(self.names)
    
    def name_to_tensor(self, name):
        # Возвращаем просто список индексов!
        indices = [all_letters.find(letter) for letter in name]
        return torch.LongTensor(indices) # Тип Long обязателен для Embedding
    
    
    def __getitem__(self, idx):
        return self.names[idx], self.labels[idx]

In [3]:
dataset = NameDataset()
print(f'Всего фамилий: {len(dataset)}')
print(f'Примеры: {dataset.names[:5]}')

Всего фамилий: 20074
Примеры: ['Khoury', 'Nahas', 'Daher', 'Gerges', 'Nazari']


In [4]:
n = len(dataset)
train_size = int(n*0.8)
val_size = int(n*0.1)
teset_size = int(n*0.1)
train_indices = range(0, train_size)
val_indices = range(train_size, train_size + val_size)
test_indices = range(train_size + val_size, n)
train_ds = data.Subset(dataset, train_indices)
val_ds = data.Subset(dataset, val_indices)
test_ds = data.Subset(dataset, test_indices)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
def collate_fn(batch):
    names, labels = zip(*batch)
    name_tensors = [dataset.name_to_tensor(name) for name in names]

    padded = pad_sequence(name_tensors, batch_first=True)
    
    labels = torch.LongTensor(labels)
    
    return padded, labels

In [6]:
train_loader = data.DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = data.DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = data.DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, emb_size, hidden_size, output_size, n_layers=2):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.emb = nn.Embedding(input_size, embedding_dim=emb_size)
        self.dropout_emb = nn.Dropout(0.3)
        
        self.rnn = nn.GRU(
            emb_size,
            hidden_size,
            bidirectional=True,
            batch_first=True,
            num_layers=n_layers,
            dropout=0.3 if n_layers > 1 else 0,  # dropout только если слоёв > 1
        )
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size * 2, output_size)  # *2 для bidirectional!
    
    def forward(self, x):
        
        x = self.emb(x) 
        x = self.dropout_emb(x)
        
        output, hidden = self.rnn(x)
        
        hidden_forward = hidden[-2] 
        hidden_backward = hidden[-1]
        
        x = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        x = self.dropout(x)
        x = self.fc(x) 
        
        return x

In [ ]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
model = RNN(
    input_size= n_letters,
    emb_size=16,
    hidden_size=128,
    output_size=len(dataset.classes),
    n_layers=1
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',     
    factor=0.5,     
    patience=5,      
    min_lr=1e-6,   
)

In [ ]:
from tqdm import tqdm


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    
    avg_loss = val_loss / len(val_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


epochs = 15
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
    for inputs, targets in loop:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        
        predict = model(inputs)
        loss = criterion(predict, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
    
        train_loss += loss.item()
        
        # Accuracy
        _, predicted = torch.max(predict, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()
        
        loop.set_postfix(loss=loss.item(), acc=f'{100*correct/total:.1f}%')
    
    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    train_acc = 100 * correct / total
    history['train_acc'].append(train_acc)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)  # Обновляем LR на основе val_loss
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    
    
    print(f'\nEpoch {epoch+1}/{epochs}:')
    print(f'  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%')
    

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✓ Saved best model (val_loss: {val_loss:.4f})')
    
    print('-' * 60)
print(f'Лучший val_loss: {best_val_loss:.4f}')

Epoch 1/15:   0%|          | 0/251 [00:00<?, ?it/s]

Epoch 1/15: 100%|██████████| 251/251 [00:03<00:00, 65.21it/s, acc=47.2%, loss=1.44]



Epoch 1/15:
  Train Loss: 1.7324 | Train Acc: 47.20%
  Val Loss:   0.8319 | Val Acc:   76.88%
  ✓ Saved best model (val_loss: 0.8319)
------------------------------------------------------------


Epoch 2/15: 100%|██████████| 251/251 [00:03<00:00, 67.89it/s, acc=57.5%, loss=1.28]



Epoch 2/15:
  Train Loss: 1.3919 | Train Acc: 57.51%
  Val Loss:   0.7428 | Val Acc:   79.47%
  ✓ Saved best model (val_loss: 0.7428)
------------------------------------------------------------


...


Epoch 13/15:
  Train Loss: 0.8623 | Train Acc: 73.63%
  Val Loss:   1.2041 | Val Acc:   71.10%
------------------------------------------------------------


Epoch 14/15: 100%|██████████| 251/251 [00:03<00:00, 69.81it/s, acc=74.0%, loss=0.786]



Epoch 14/15:
  Train Loss: 0.8501 | Train Acc: 74.02%
  Val Loss:   1.3113 | Val Acc:   66.82%
------------------------------------------------------------


Epoch 15/15: 100%|██████████| 251/251 [00:03<00:00, 67.28it/s, acc=74.0%, loss=0.572]



Epoch 15/15:
  Train Loss: 0.8490 | Train Acc: 73.96%
  Val Loss:   1.2106 | Val Acc:   70.25%
------------------------------------------------------------
Лучший val_loss: 0.7293


In [11]:
with torch.no_grad():
    model.load_state_dict(torch.load('best_model.pt'))
    test_loss, test_acc = validate(model, test_loader, criterion, device)
    print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%')

Test Loss: 2.1715 | Test Acc: 65.09%


In [ ]:
def predict(model, name):
    model.eval() 
    
    with torch.no_grad(): 
        indices = [all_letters.find(c) for c in name]
        tensor = torch.LongTensor(indices).to(device)
        
        tensor = tensor.unsqueeze(0) 
        
        output = model(tensor) 

        probs = torch.nn.functional.softmax(output, dim=1)
        
        top_value, top_index = probs.topk(1)
        
        category_index = top_index.item()
        probability = top_value.item()
        
        predicted_class = dataset.classes[category_index]
        
        return predicted_class, probability

my_names = ["Putin", "Obama", "Satoshi", "Muller", "Ronaldo", "Ruseva","Schwarzenegger",
    "Miyazaki", 
    "O'Neil",         
    "Kowalski",
    "Nguyen",
    "Messi",
    "Neymar"]

print(f"{'ИМЯ':<15} {'ПРЕДСКАЗАНИЕ':<15} {'УВЕРЕННОСТЬ'}")
print("-" * 45)

for name in my_names:
    pred, prob = predict(model, name)
    print(f"{name:<15} {pred:<15} {prob:.2%}")

ИМЯ             ПРЕДСКАЗАНИЕ    УВЕРЕННОСТЬ
---------------------------------------------
Putin           English         18.52%
Obama           Japanese        54.90%
Satoshi         Japanese        47.19%
Muller          English         30.84%
Ronaldo         English         31.65%
Ruseva          Russian         81.67%
Schwarzenegger  German          44.86%
Miyazaki        Japanese        40.29%
O'Neil          English         13.45%
Kowalski        Greek           37.47%
Nguyen          Russian         26.34%
Messi           Greek           71.97%
Neymar          English         33.95%
